In [ ]:
import pandas as pd
from transformers import pipeline

C:\Users\vishw\anaconda3\envs\mlenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
reviews = pd.read_parquet(r'C:\Users\vishw\OneDrive\Documents\DataMiningJobReview\reviews.parquet')  # your real file

In [ ]:
import torch
device = 0 if torch.cuda.is_available() else -1
print(f"Using {'GPU: ' + torch.cuda.get_device_name(0) if device == 0 else 'CPU'}")
 
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device,
)

In [8]:
CANDIDATE_LABELS = [
    "burnout and exhaustion",
    "compassion fatigue from difficult clients or students",
    "frustration with management or leadership",
    "dissatisfaction with pay",
    "high staff turnover and instability",
    "overall job satisfaction",
    "supportive coworkers and teamwork",
    "pride and sense of purpose",
    "healthy work-life balance",
]

In [11]:
reviews["full_text"] = reviews["pros"].fillna("") + " " + reviews["cons"].fillna("")
texts = reviews["full_text"].tolist()
 
BATCH_SIZE = 8  # tuned for 6GB VRAM laptop GPUs (e.g. RTX 4050); try 16 if it runs comfortably, drop to 4 if you hit "out of memory"
all_scores = []
for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i:i + BATCH_SIZE]
    batch_results = classifier(batch, CANDIDATE_LABELS, multi_label=True)
    # classifier returns a single dict if batch has 1 item, list otherwise
    if isinstance(batch_results, dict):
        batch_results = [batch_results]
    for result in batch_results:
        all_scores.append(dict(zip(result["labels"], result["scores"])))
    print(f"Classified {min(i + BATCH_SIZE, len(texts))}/{len(texts)}")
 
scores_df = pd.DataFrame(all_scores)
scores_df.columns = [c.replace(" ", "_") + "_score" for c in scores_df.columns]
 
reviews_scored = pd.concat([reviews.reset_index(drop=True), scores_df], axis=1)
 

Classified 8/1543
Classified 16/1543
Classified 24/1543
Classified 32/1543
Classified 40/1543
Classified 48/1543
Classified 56/1543
Classified 64/1543
Classified 72/1543


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Classified 80/1543
Classified 88/1543
Classified 96/1543
Classified 104/1543
Classified 112/1543
Classified 120/1543
Classified 128/1543
Classified 136/1543
Classified 144/1543
Classified 152/1543
Classified 160/1543
Classified 168/1543
Classified 176/1543
Classified 184/1543
Classified 192/1543
Classified 200/1543
Classified 208/1543
Classified 216/1543
Classified 224/1543
Classified 232/1543
Classified 240/1543
Classified 248/1543
Classified 256/1543
Classified 264/1543
Classified 272/1543
Classified 280/1543
Classified 288/1543
Classified 296/1543
Classified 304/1543
Classified 312/1543
Classified 320/1543
Classified 328/1543
Classified 336/1543
Classified 344/1543
Classified 352/1543
Classified 360/1543
Classified 368/1543
Classified 376/1543
Classified 384/1543
Classified 392/1543
Classified 400/1543
Classified 408/1543
Classified 416/1543
Classified 424/1543
Classified 432/1543
Classified 440/1543
Classified 448/1543
Classified 456/1543
Classified 464/1543
Classified 472/1543
Cla

In [13]:
reviews_scored["invisible_labor_score"] = (
    reviews_scored["burnout_and_exhaustion_score"] * 1.0
    + reviews_scored["compassion_fatigue_from_difficult_clients_or_students_score"] * 0.75
    + reviews_scored["frustration_with_management_or_leadership_score"] * 0.5
    + reviews_scored["dissatisfaction_with_pay_score"] * 0.5
    + reviews_scored["high_staff_turnover_and_instability_score"] * 0.5
    - reviews_scored["pride_and_sense_of_purpose_score"] * 0.5
)
 
reviews_scored["positive_experience_score"] = (
    reviews_scored["overall_job_satisfaction_score"] * 1.0
    + reviews_scored["supportive_coworkers_and_teamwork_score"] * 0.75
    + reviews_scored["pride_and_sense_of_purpose_score"] * 0.75
    + reviews_scored["healthy_work-life_balance_score"] * 0.5
)
 
reviews_scored.to_csv(r'C:\Users\vishw\OneDrive\Documents\classified_reviews_zeroshot.csv', index=False)
print("\nSaved classified_reviews_zeroshot.csv")


Saved classified_reviews_zeroshot.csv


In [23]:
classified_reviews = pd.read_csv(r'C:\Users\vishw\OneDrive\Documents\classified_reviews_zeroshot.csv')

In [25]:
classified_reviews.head()

,company_name,department,occupation,page,rating,date,title,job_title,employment_status,location,...,overall_job_satisfaction_score,healthy_work-life_balance_score,pride_and_sense_of_purpose_score,compassion_fatigue_from_difficult_clients_or_students_score,frustration_with_management_or_leadership_score,burnout_and_exhaustion_score,high_staff_turnover_and_instability_score,dissatisfaction_with_pay_score,invisible_labor_score,positive_experience_score
0,HealthTrust Workforce Solutions,Healthcare,Nurse,1,4.0,"Jan 14, 2026",I love healthtrust,"Registered nurse, bsn",Current employee,NaN,...,0.989679,0.923319,0.746604,0.116621,0.041167,0.014943,0.006181,0.006165,-0.244137,2.759285
1,HealthTrust Workforce Solutions,Healthcare,Nurse,1,3.0,"Jan 3, 2026",Okay,"Registered nurse, bsn",Current employee,"San Antonio, TX",...,0.068793,0.148142,0.136294,0.004567,0.152505,0.001075,0.315970,0.520575,0.430878,0.341078
2,HealthTrust Workforce Solutions,Healthcare,Nurse,1,4.0,"Aug 20, 2025",Not a bad company to work for,"Registered nurse, bsn","Current employee, more than 3 years",Salt Lake City,...,0.640716,0.978437,0.099966,0.057352,0.534291,0.011752,0.233246,0.886876,0.831989,1.206703
3,HealthTrust Workforce Solutions,Healthcare,Nurse,1,4.0,"May 12, 2025",Peace at work,"Registered nurse, bsn","Current employee, more than 3 years","Richmond, VA",...,0.063112,0.116940,0.002046,0.015239,0.963927,0.201550,0.000127,0.997475,1.192720,0.123519
4,HealthTrust Workforce Solutions,Healthcare,Nurse,1,5.0,"Apr 10, 2025",Flexible per diem,"Registered nurse, bsn",Current employee,"Dallas, TX",...,0.027785,0.288605,0.019905,0.002574,0.071380,0.001633,0.448733,0.097827,0.302581,0.189539
